# Sesi 9 — Spam Classifier dengan Fine-tuning BERT

> ⚠️ **Jalankan notebook ini di Google Colab** (Runtime → Change runtime type → GPU disarankan).
> Notebook ini membutuhkan akses ke Hugging Face Hub untuk mengunduh bobot pretrained `bert-base-uncased`.
>
> Setelah menjalankan seluruh cell, catat angka akurasi/precision/recall/F1 dari Bagian 5 (fine-tuning penuh)
> dan Bagian 6 (perbandingan Naive Bayes vs BERT frozen) — angka tersebut yang dipakai untuk melengkapi
> laporan/blog, bukan angka ilustrasi dari slide.

Alur: **Data berlabel → Tokenizer BERT → Fine-tuning → Evaluasi → Deploy/Predict**


## 0. Instalasi & Import

Sengaja **tidak** meng-upgrade `torch`/`pandas` — versi bawaan Colab sudah cocok dengan `torchvision` dan library CUDA lain yang sudah terpasang. Meng-upgrade torch tanpa upgrade torchvision yang sepasang menyebabkan error `torchvision::nms does not exist`.

In [ ]:
!pip install -q transformers datasets scikit-learn --upgrade

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## 1. Siapkan Data

Dataset: **SMS Spam Collection v.1** (Almeida et al., 2011) — rilis resmi berisi 5.574 pesan SMS
berbahasa Inggris berlabel ham/spam. Mirror GitHub yang dipakai di sini berisi 5.572 baris
(selisih 2 baris dari rilis resmi, kemungkinan perbedaan parsing pada mirror).
Kolom: `text` + `label` (0 = ham, 1 = spam).

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
df = df.drop_duplicates().reset_index(drop=True)
df["label_id"] = (df["label"] == "spam").astype(int)

print("Total setelah hapus duplikat:", len(df))
print(df["label"].value_counts())
df.head()


In [ ]:
# Split stratified: 80% train, 10% val, 10% test
train_val, test_df = train_test_split(
    df, test_size=0.1, stratify=df["label_id"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1111, stratify=train_val["label_id"], random_state=SEED
)

print("train:", len(train_df), train_df["label"].value_counts().to_dict())
print("val  :", len(val_df), val_df["label"].value_counts().to_dict())
print("test :", len(test_df), test_df["label"].value_counts().to_dict())


## 2. Tokenisasi

Menggunakan `bert-base-uncased`. `max_length=128` sudah cukup karena rata-rata pesan SMS pendek
(rata-rata ham 71,5 karakter, spam 138,7 karakter).

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

train_enc = tokenize(train_df["text"])
val_enc = tokenize(val_df["text"])
test_enc = tokenize(test_df["text"])


In [ ]:
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = SpamDataset(train_enc, train_df["label_id"])
val_dataset = SpamDataset(val_enc, val_df["label_id"])
test_dataset = SpamDataset(test_enc, test_df["label_id"])


## 3. Load Model & Tambah Kepala Klasifikasi

`AutoModelForSequenceClassification` otomatis menambahkan Dense layer di atas vektor `[CLS]`
untuk klasifikasi 2 kelas (ham=0, spam=1).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


## 4. Fine-tuning

Hyperparameter mengikuti rekomendasi materi: `lr=2e-5`, `batch_size=16`, `epochs=3`.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


In [ ]:
train_result = trainer.train()
train_result


## 5. Evaluasi di Test Set

⚠️ **Isi angka di bawah ini dengan hasil aktual setelah menjalankan cell ini** — jangan pakai placeholder.

In [ ]:
test_metrics = trainer.evaluate(test_dataset)
test_metrics


In [ ]:
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = test_df["label_id"].values

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["ham", "spam"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Test Set")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


### Ringkasan Metrik

| Metrik | Nilai |
|---|---|
| Accuracy | 0,9923 |
| Precision (spam) | 0,9692 |
| Recall (spam) | 0,9692 |
| F1 (spam) | 0,9692 |

Confusion matrix (test set, 517 pesan): 450 ham diprediksi benar, 2 ham salah diprediksi spam,
2 spam salah diprediksi ham, 63 spam diprediksi benar.


## 6. Perbandingan dengan Pendekatan Lain

Saat menelusuri materi bootcamp lebih jauh, ditemukan notebook referensi resmi yang
mengimplementasikan tugas ini dengan cara **berbeda dari fine-tuning penuh di atas**:

1. **Baseline klasik**: CountVectorizer + Naive Bayes.
2. **BERT sebagai feature extractor (bukan fine-tuning penuh)**: seluruh parameter BERT **dibekukan**
   (`requires_grad = False`), hanya head klasifikasi custom (Linear 768→512→2) yang dilatih.

Ini menarik karena slide teori menyebut *"seluruh parameter BERT juga ikut di-update (end-to-end)"* —
padahal notebook resmi bootcamp justru membekukan BERT-nya. Supaya perbandingan adil, kedua pendekatan
ini dijalankan ulang di bawah **memakai split data yang sama persis** dengan fine-tuning penuh di atas
(`train_df`, `val_df`, `test_df` — dedup, stratified 80/10/10, `random_state=42`), bukan sekadar
mengutip angka dari notebook referensi (yang split-nya beda: tanpa dedup, rasio 70/15/15).

> Sebagai konteks kualitatif saja (bukan pembanding langsung, karena beda split & tanpa dedup):
> notebook referensi resmi mendapati Naive Bayes (F1 spam **0.97**) sedikit mengungguli BERT-frozen
> (F1 spam **0.94**) pada test set mereka.

### 6a. Baseline: CountVectorizer + Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(train_df["text"])

nb_model = MultinomialNB().fit(X_train_counts, train_df["label_id"])
nb_preds = nb_model.predict(count_vect.transform(test_df["text"]))

nb_precision, nb_recall, nb_f1, _ = precision_recall_fscore_support(
    test_df["label_id"], nb_preds, average="binary"
)
nb_acc = accuracy_score(test_df["label_id"], nb_preds)
print(f"Naive Bayes — accuracy={nb_acc:.4f}  precision={nb_precision:.4f}  recall={nb_recall:.4f}  f1={nb_f1:.4f}")


### 6b. BERT sebagai Feature Extractor (Frozen) + Head Custom

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class BERT_Arch(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, 2)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_ids, attention_mask):
        _, cls_hs = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        x = self.relu(self.fc1(cls_hs))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.log_softmax(x)

bert_frozen = AutoModel.from_pretrained(MODEL_NAME)
for param in bert_frozen.parameters():
    param.requires_grad = False  # ini bagian kuncinya: BERT dibekukan, tidak ikut fine-tune

frozen_model = BERT_Arch(bert_frozen).to(trainer.args.device)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader

class_wts = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_df["label_id"]), y=train_df["label_id"]
)
weights = torch.tensor(class_wts, dtype=torch.float).to(trainer.args.device)
criterion = nn.NLLLoss(weight=weights)
frozen_optimizer = torch.optim.AdamW(frozen_model.parameters(), lr=1e-3)

frozen_train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
frozen_test_loader = DataLoader(test_dataset, batch_size=32)

FROZEN_EPOCHS = 5
frozen_model.train()
for epoch in range(FROZEN_EPOCHS):
    epoch_loss = 0
    for batch in frozen_train_loader:
        frozen_optimizer.zero_grad()
        input_ids = batch["input_ids"].to(trainer.args.device)
        attention_mask = batch["attention_mask"].to(trainer.args.device)
        labels = batch["labels"].to(trainer.args.device)
        out = frozen_model(input_ids, attention_mask)
        loss = criterion(out, labels)
        loss.backward()
        frozen_optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{FROZEN_EPOCHS} — loss: {epoch_loss/len(frozen_train_loader):.4f}")


In [ ]:
frozen_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in frozen_test_loader:
        input_ids = batch["input_ids"].to(trainer.args.device)
        attention_mask = batch["attention_mask"].to(trainer.args.device)
        out = frozen_model(input_ids, attention_mask)
        all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

fz_precision, fz_recall, fz_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")
fz_acc = accuracy_score(all_labels, all_preds)
print(f"Frozen-BERT — accuracy={fz_acc:.4f}  precision={fz_precision:.4f}  recall={fz_recall:.4f}  f1={fz_f1:.4f}")


### Tabel Perbandingan

Semua tiga model dievaluasi di **test set yang sama** (`test_df`, dedup, split 80/10/10, seed 42):

| Model | Accuracy | Precision (spam) | Recall (spam) | F1 (spam) |
|---|---|---|---|---|
| Naive Bayes (baseline) | 0,9845 | 0,9524 | 0,9231 | 0,9375 |
| BERT frozen + head (pendekatan resmi bootcamp) | 0,9845 | 0,8904 | 1,0000 | 0,9420 |
| BERT full fine-tuning (Bagian 3–5 di atas) | 0,9923 | 0,9692 | 0,9692 | 0,9692 |

Hasilnya: fine-tuning penuh memang unggul di split ini pada semua metrik. Naive Bayes tetap kompetitif
mengingat kesederhanaannya. Frozen-BERT punya recall sempurna (1,0000, artinya tidak ada spam yang lolos)
tapi precision paling rendah (lebih banyak ham yang salah ditandai spam). Ini trade-off yang berbeda dari
dua model lainnya, bukan sekadar performa yang lebih buruk secara keseluruhan.

## 7. Demo Inferensi

Menguji model (full fine-tuning) pada teks baru.

In [ ]:
def predict_spam(texts, model=model, tokenizer=tokenizer):
    model.eval()
    device = next(model.parameters()).device
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1)[:, 1]  # probabilitas kelas spam
    return [
        {"text": t, "label": "SPAM" if p >= 0.5 else "HAM", "p_spam": round(p.item(), 4)}
        for t, p in zip(texts, probs)
    ]

demo_texts = [
    "Congratulations! You've won a free iPhone, click the link to claim now.",
    "Hey, what time is the meeting tomorrow?",
    "URGENT loan approved in 5 minutes, WhatsApp us now!",
    "Can you send me the revised file, please?",
]

for r in predict_spam(demo_texts):
    print(r)


**Hasil aktual (4/4 dijalankan):**

| Teks | Prediksi | p_spam |
|---|---|---|
| "Congratulations! You've won a free iPhone, click the link to claim now." | SPAM | 0,9988 |
| "Hey, what time is the meeting tomorrow?" | HAM | 0,0008 |
| "URGENT loan approved in 5 minutes, WhatsApp us now!" | **HAM** | 0,0006 |
| "Can you send me the revised file, please?" | HAM | 0,0058 |

Baris ketiga meleset. Teks itu ditulis untuk terdengar seperti spam pinjaman, tapi model memprediksinya
HAM dengan keyakinan tinggi. Dugaan penyebabnya: dataset SMS Spam Collection berasal dari sekitar 2011,
sebelum "WhatsApp" jadi istilah umum di teks spam berbahasa Inggris, sehingga frasa ini kemungkinan besar
di luar distribusi data training. Ini pengingat bahwa model setinggi apa pun akurasinya di test set tetap
bisa salah pada pola yang tidak ada di data latihnya.

## 8. Simpan Model

```python
model.save_pretrained("bert-spam-classifier")
tokenizer.save_pretrained("bert-spam-classifier")
```

Catatan: file bobot model (`.bin`/`.safetensors`, mirip `.pkl`) **tidak** ikut di-push ke GitHub
(masuk `.gitignore`) karena ukurannya besar — cukup kode + README + hasil evaluasi yang di-commit.


---
*Bootcamp: rubythalib.ai AI Engineer Bootcamp, Sesi 9. Mentor: Daniel Syahputra.*
*Statistik dataset dan kode perbandingan tambahan (Bagian 6) disusun mandiri untuk keperluan verifikasi.*
